# Kapitel 1 — Daten lesen · Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch01_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Aufgabe greift die Aufgabenstellung des Buchs auf. Die Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python.

**Konvention.** Wie im Buch teilen Varianz, Standardabweichung, z-Wert und Kovarianz durch $n$ (`ddof=0`), sofern nicht ausdrücklich anders angegeben.

**Inhalt.** Daten · Die Berechnungen der Lektionen · Aufgaben 1.1 bis 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennungen (D0.1, D1.1…) sind die des Buchs und der CSV-Dateien.

Variablen- und Spaltennamen im Code bleiben französisch: `logements` / `logement` = Wohnungen / Wohnung, `surface_m2` = Wohnfläche in m², `prix_kEUR` = Preis in k€, `nord` / `sud` = Viertel Nord / Süd, `registre` = Verkaufsregister, `vente` = Verkauf, `pieces` = Zimmer, `dpe` = Energieeffizienzklasse, `delai_jours` = Verkaufsdauer in Tagen, `etage` = Stockwerk, `delais_agences` = Verkaufsdauern der Agenturen, `tilleuls` / `port` = die beiden Wohnanlagen (Les Tilleuls, Le Port), `loyers` = Mieten, `moyenne` = Mittelwert, `ecart(s)` = Abweichung(en).

In [ ]:
# Daten von Kapitel 1, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen der Einleitung
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — Verkaufspreise der T2-Wohnungen (zwei Zimmer, k€), Lektion 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — Verkaufsregister des Viertels Süd, Lektionen 1.2 bis 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — Preis pro m² nach Stockwerk (k€/m²), Lektion 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — Verkaufsdauern von drei Agenturen (Tage), Aufgabe 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — Monatsmieten der Studios (€), Aufgabe 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Bereitgestellte Werkzeuge: Sie müssen sie nicht ändern

def quartiles_livre(valeurs):
    """Q1 und Q3 nach der Regel des Buchs (Gleichung 1.5): Ränge ⌈n/4⌉ und ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # n/4 auf die nächste ganze Zahl aufgerundet
    rang_q3 = -(-3 * n // 4)      # 3n/4 auf die nächste ganze Zahl aufgerundet
    return x[rang_q1 - 1], x[rang_q3 - 1]   # Rang 1 = Position 0 in Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Eine Punktreihe pro Datenreihe, wie in den Abbildungen von Lektion 1.1.
    bandes: Wörterbuch {Name: (Mittelwert, Standardabweichung)}, um Mittelwert ± Standardabweichung zu zeichnen."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 1.1 bis 1.5. Führen Sie sie aus und vergleichen Sie mit dem Buch. Hier werden `np.std` und `np.var` mit `ddof=0` aufgerufen, also mit einer Division durch $n$, wie im Kapitel.

In [ ]:
# Lektion 1.1 — Mittelwert, Varianz und Standardabweichung (Division durch n)
for nom, serie in [('Nord', nord), ('Süd', sud)]:
    print(f'{nom}: Mittelwert = {np.mean(serie):g} k€, Varianz = {np.var(serie, ddof=0):g} k€², '
          f'Standardabweichung = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Lektion 1.2 — Häufigkeiten je Energieeffizienzklasse (ordinale qualitative Variable)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lektion 1.3 — Mittelwert, Median und Quartile der Verkaufsdauern
d = registre['delai_jours']
print('Mittelwert:', d.mean(), ' Median:', d.median())
print('ohne S13: Mittelwert', d[registre['vente'] != 'S13'].mean(),
      ' Median', d[registre['vente'] != 'S13'].median())
print('Quartile, Regel des Buchs (Gleichung 1.5):', quartiles_livre(d))
print('Quartile, NumPy-Standard (Interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lektion 1.4 — standardisierte Werte (Gleichung 1.6, Standardabweichung mit Division durch n)
for nom, serie, vente in [('Nord', nord, 188), ('Süd', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom}: z der Reihe = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z des Verkaufs zu {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Lektion 1.5 — Kovarianz und Korrelation der fünf T2-Wohnungen im Süden (Division durch n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('Kovarianz:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r mit np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r Stockwerk / Preis pro m²:', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Aufgabe 1.1 — Ein Diagramm lesen

*Daten D1.E1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1).* Das folgende Diagramm gibt das aus Aufgabe 1.1 des Buchs wieder.

In [ ]:
points_par_rangee({f'Agentur {a}': v for a, v in delais_agences.items()},
                  'Verkaufsdauer (Tage)')
plt.xlim(0, 60)
plt.show()

**a) und b)** Schätzen Sie nach Augenmaß die mittlere Verkaufsdauer jeder Agentur und ordnen Sie die Agenturen dann von der am wenigsten zur am stärksten gestreuten.

In [ ]:
moyenne_estimee = {'P': 30, 'Q': 30, 'R': 30}   # in Tagen
ordre_dispersion = ['P', 'Q', 'R']

In [ ]:
# Prüfung (Division durch n)
for a, v in delais_agences.items():
    print(f'Agentur {a}: Mittelwert = {np.mean(v):g} T, Standardabweichung = {np.std(v, ddof=0):.2f} T')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Reihenfolge der Streuung:', 'richtig' if ordre_dispersion == ordre_reel else 'bitte prüfen')

**c)** Zählen Sie für Q und für R die Verkäufe, die höchstens 2 Tage vom Mittelwert entfernt abgeschlossen wurden.

In [ ]:
proches_Q = 1
proches_R = 3

In [ ]:
# Prüfung
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agentur {a}: noch nicht ausgefüllt.')
    else:
        print(f"Agentur {a}: {'richtig' if la_votre == vrai else 'bitte prüfen'}")

### Lösung zu Aufgabe 1.1

**a)** Die drei Reihen sind symmetrisch um 30 Tage. Die mittlere Verkaufsdauer beträgt für jede Agentur **30 Tage**, wie die Rechnung bestätigt: (28 + 29 + 30 + 31 + 32) / 5 = 150 / 5 = 30, ebenso für Q und R.

**b)** Von der am wenigsten zur am stärksten gestreuten: **P, Q, R**. Die Punkte von P liegen zwischen 28 und 32 Tagen. Die von Q verteilen sich gleichmäßig von 20 bis 40 Tagen. Die von R reichen von 10 bis 50 Tagen. Standardabweichungen (Division durch $n$): P ≈ 1,41 T; Q ≈ 7,07 T; R ≈ 12,71 T.

**c)** Q: **1** Verkauf (30 Tage). R: **3** Verkäufe (28, 30 und 32 Tage). Dennoch hat R die größte Standardabweichung. Seine beiden extremen Verkäufe, nach 10 und 50 Tagen, liegen 20 Tage vom Mittelwert entfernt. Ihre Quadrate betragen je 400: Die Quadratsumme von R beträgt 400 + 4 + 0 + 4 + 400 = 808, gegenüber 100 + 25 + 0 + 25 + 100 = 250 für Q. Die Standardabweichung fasst alle Abweichungen zusammen, und das Quadrieren gibt den größten ein hohes Gewicht. Drei Verkäufe nahe der Mitte gleichen zwei sehr weit entfernte Verkäufe nicht aus.

*Erwartete Elemente:* die Reihenfolge P, Q, R; die beiden Zählungen; die Rolle der beiden Extremwerte und der Quadrate in der Rechnung.

## Aufgabe 1.2 — Zwei Wohnanlagen, dieselbe mittlere Miete

*Daten D1.E2. Begriffe: Mittelwert, Varianz und Standardabweichung (Lektion 1.1).* Camille kann höchstens 570 € im Monat ausgeben. Die Fragen a) bis d) lösen Sie zuerst auf Papier, mit der Tabelle aus Aufgabe 1.2 des Buchs. Dieses Notebook dient dazu, Ihre Antworten zu prüfen, dann die Standardabweichung zu programmieren (Frage e) und zwei Konventionen zu vergleichen (Frage f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monatsmiete eines Studios (€)")
plt.axvline(570, color='black', linestyle=':', label='Budget von Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Die Mieten ablesen

Wie viele Studios kann Camille in jeder Wohnanlage mit 570 € mieten, und wie viele mit 620 €?

In [ ]:
accessibles = {
    570: {'Les Tilleuls': 0, 'Le Port': 2},
    620: {'Les Tilleuls': 6, 'Le Port': 4},
}

In [ ]:
# Prüfung: `<=` liefert Wahr/Falsch-Werte, np.sum zählt die „Wahr“-Werte
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if la_votre == vrai else 'bitte prüfen')
        print(f'Budget {budget} €, {nom}: {etat}')

### Lösung zu 1.2 a)

**Beim Lesen der Liste.** Mit 570 € findet Camille in Les Tilleuls **kein** Studio: Das günstigste kostet 585 €. In Le Port sind **zwei** von sechs Studios für sie erreichbar, zu 520 und 560 €. Die Wohnanlage, in der sie eine Chance hat, ist also die mit den am stärksten gestreuten Mieten. Mit 620 € kehrt sich die Lage um: Alle **sechs** Studios in Les Tilleuls sind für sie erreichbar, gegenüber **vier** von sechs in Le Port (520, 560, 590 und 620 €). Die beiden gleichen Mittelwerte ließen von diesen Unterschieden nichts ahnen.

### b) Die beiden Mittelwerte

**Auf Papier.** Les Tilleuls: 585 + 590 + 600 + 605 + 605 + 615 = 3600, und 3600 / 6 = **600 €**. Le Port: 520 + 560 + 590 + 620 + 640 + 670 = 3600, und 3600 / 6 = **600 €**.

**In Python.** `np.mean` bildet dieselbe Summe und dann dieselbe Division.

In [ ]:
moyenne_tilleuls_main = 600
moyenne_port_main = 600

In [ ]:
# Prüfung
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom}: bitte füllen Sie die vorherige Zelle aus.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom}: {'richtig' if ok else 'bitte prüfen'} (Ihr Wert: {la_votre} €)")

### c) Varianz und Standardabweichung von Hand (Division durch $n = 6$)

**Les Tilleuls** ($\bar{x} = 600$ €)

| $x_i$ (€) | 585 | 590 | 600 | 605 | 605 | 615 | Summe |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −15 | −10 | 0 | +5 | +5 | +15 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 225 | 100 | 0 | 25 | 25 | 225 | 600 |

Varianz: 600 / 6 = **100 €²**. Standardabweichung: √100 = **10 €**.

**Le Port** ($\bar{x} = 600$ €)

| $x_i$ (€) | 520 | 560 | 590 | 620 | 640 | 670 | Summe |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −80 | −40 | −10 | +20 | +40 | +70 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 6400 | 1600 | 100 | 400 | 1600 | 4900 | 15 000 |

Varianz: 15 000 / 6 = **2500 €²**. Standardabweichung: √2500 = **50 €**.

*Zwei Kontrollen unterwegs.* In jeder Tabelle muss die Zeile der Abweichungen zusammen 0 ergeben (Lektion 1.1). Außerdem muss die Standardabweichung in derselben Größenordnung bleiben wie die Abweichungen der Tabelle: 10 € bei Abweichungen von 0 bis 15 €, 50 € bei Abweichungen von 10 bis 80 €.

**In Python.** `np.var(..., ddof=0)` und `np.std(..., ddof=0)` teilen durch $n$.

In [ ]:
variance_tilleuls_main = 100   # in €²
variance_port_main = 2500      # in €²
ecart_type_tilleuls_main = 10  # in €
ecart_type_port_main = 50      # in €

In [ ]:
# Prüfung (Division durch n: ddof=0)
vos_valeurs = {
    'Varianz Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'Varianz Port': (variance_port_main, np.var(port, ddof=0)),
    'Standardabweichung Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'Standardabweichung Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom}: noch nicht ausgefüllt.')
    else:
        print(f"{nom}: {'richtig' if np.isclose(la_votre, attendu) else 'bitte prüfen'}")

### d) Interpretieren

*Beispielantwort.* In Les Tilleuls weichen die Mieten gewöhnlich um etwa 10 € von 600 € ab: Sie sind fast alle gleich. In Le Port beträgt die übliche Abweichung etwa 50 €: Dort gibt es Studios, die deutlich günstiger, und solche, die deutlich teurer als der Mittelwert sind. Die Standardabweichung reicht nicht aus, um Frage a) zu beantworten. Sie beschreibt die Streuung im Allgemeinen, sagt aber nicht, ob ein bestimmtes Studio 570 € oder weniger kostet. Eine Standardabweichung von 50 € um 600 € macht eine Miete unter 570 € plausibel. Nur die Liste erlaubt es zu zählen, dass es genau zwei sind, zu 520 und 560 €.

*Erwartete Elemente:* ein Satz pro Wohnanlage, mit Einheit; die Unterscheidung zwischen einem Streuungsmaß und der Frage „Passt ein Studio ins Budget?“; der Rückgriff auf die Mietliste.

### e) Die Standardabweichung programmieren

Vervollständigen Sie die Funktion nach Gleichung 1.3 des Buchs, mit einer Division durch $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Verwenden Sie weder `np.std` noch `np.var`: Ziel ist, jeden Schritt selbst zu schreiben.

In [ ]:
# Aufgabe 1.2 e)
def ecart_type(valeurs):
    """Standardabweichung einer Reihe, mit Division durch n (Gleichung 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = np.sum(x) / n               # Gleichung 1.1
    ecarts = x - moyenne                  # eine Subtraktion pro Wert
    variance = np.sum(ecarts ** 2) / n    # Gleichung 1.2
    return np.sqrt(variance)              # zurück zu Euro

Jede Zeile der Funktion entspricht einer Spalte der Tabelle aus Frage c): die Abweichungen, ihre Quadrate, der Mittelwert dieser Quadrate, dann die Quadratwurzel, um zu Euro zurückzukehren. Häufige Fehler: die Wurzel vor dem Mittelwert ziehen; `np.sum(ecarts) / n` berechnen, was immer 0 ergibt (Lektion 1.1).

In [ ]:
# Prüfung: Führen Sie diese Zelle aus, ohne sie zu ändern
if ecart_type(tilleuls) is None:
    print('Die Funktion gibt noch None zurück: Bitte vervollständigen Sie sie.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'richtig' if np.isclose(obtenu, attendu) else f'bitte prüfen (erwartet: {attendu})'
        print(f'{nom}: {obtenu:.2f} € → {etat}')

### f) Vertiefung: durch $n$ oder durch $n - 1$ teilen

Standardmäßig teilt die pandas-Methode `.std()` die Quadratsumme durch $n - 1$ (`ddof=1`). NumPy teilt standardmäßig durch $n$ (`ddof=0`), die Konvention des Kapitels.

**Auf Papier, mit Division durch $n - 1 = 5$.** Tilleuls: 600 / 5 = 120, dann √120 ≈ **10,95 €**. Port: 15 000 / 5 = 3000, dann √3000 ≈ **54,77 €**.

Beide Konventionen sind richtig, beantworten aber nicht dieselbe Frage. Die Division durch $n$ beschreibt die sechs Studios selbst. Die Division durch $n - 1$ dient dazu, wenn diese sechs Studios eine größere Gesamtheit vertreten. Kapitel 6 wird diese Wahl erklären. Mit `.std(ddof=0)` liefert pandas wieder 10 und 50 €. Geben Sie in einer Rechnung immer die verwendete Konvention an.

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, Standard (Division durch n − 1):', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (Division durch n)      :', loyers.std(ddof=0).round(2).to_dict())

## Aufgabe 1.3 — Ein überraschender Verkauf?

*Daten D1.1. Begriffe: standardisierter Wert (Lektion 1.4, Gleichung 1.6).* Eine T2-Wohnung (zwei Zimmer) wurde gerade für 196 k€ verkauft. Mittelwerte und Standardabweichungen: die aus Lektion 1.1, mit Division durch $n$.

In [ ]:
z_nord = (196 - 180) / 4     # = 4
z_sud = (196 - 180) / 20     # = 0,8
prix_sud_aussi_exceptionnel = 180 + 4 * 20   # = 260 k€

In [ ]:
# Prüfung (Standardabweichung mit Division durch n)
attendus = {
    'z Nord': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Süd': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'gleichwertiger Preis Süd': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if np.isclose(la_votre, vrai) else 'bitte prüfen')
    print(f'{nom}: {etat}')

### Lösung zu Aufgabe 1.3

**a)** Im Norden: $z = (196 - 180)/4 = 16/4 =$ **4**. Im Süden: $z = (196 - 180)/20 = 16/20 =$ **0,8**. Beide Werte haben keine Einheit: Sie zählen Standardabweichungen.

**b)** Im **Norden** wäre er überraschend. In der Abbildung der standardisierten Werte in Lektion 1.4 liegt ein Wert von 4 weit jenseits aller Verkäufe im Norden, von denen keiner 1,25 übersteigt. Im Süden liegt 0,8 zwischen den Verkäufen bei 0,75 und 1,25: ein gewöhnlicher Verkauf.

**c)** Gesucht ist der Preis im Süden, dessen Wert 4 beträgt: $180 + 4 \times 20 =$ **260 k€**. Man geht von Gleichung 1.6 aus und löst nach $x$ auf: $x = \bar{x} + z\,\sigma$.

*Erwartete Elemente:* die beiden Werte; der Vergleich in Standardabweichungen statt in k€; der gleichwertige Preis von 260 k€.

## Aufgabe 1.4 — Gehören Fläche und Preis zusammen?

*Daten D0.1 (die sechs Wohnungen der Einleitung). Begriffe: Kovarianz und Korrelation (Lektion 1.5, Gleichungen 1.7 und 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Wohnfläche (m²)')
plt.ylabel('Verkaufspreis (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Die Punkte steigen von links nach rechts und liegen fast auf einer Geraden: $r$ wird **positiv und sehr nahe bei 1** sein.

**b)** Vervollständigen Sie die folgenden Zeilen. Teilen Sie überall durch $n$.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = (x - x.mean()) * (y - y.mean())          # ein Produkt pro Wohnung
covariance = np.sum(produits) / n                   # Gleichung 1.7
r = covariance / (x.std(ddof=0) * y.std(ddof=0))    # Gleichung 1.8

**Auf Papier** beträgt die mittlere Fläche 384 / 6 = 64 m² und der mittlere Preis 1334 / 6 ≈ 222,33 k€. Die Summe der sechs Produkte beträgt 9825, also eine Kovarianz von 9825 / 6 = **1637,5 m²·k€**. Die Standardabweichungen betragen etwa 22,80 m² und 72,15 k€, also $r ≈ 1637{,}5 / (22{,}80 × 72{,}15) ≈$ **0,996**.

In [ ]:
# Prüfung
if r is None or covariance is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Kovarianz:', round(covariance, 2), 'm²·k€ →',
          'richtig' if np.isclose(covariance, 1637.5) else 'bitte prüfen (erwartet: 1637,5)')
    print('r:', round(r, 4), '→', 'richtig' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'bitte prüfen')

**Eine Falle, die man kennen sollte.** `np.cov(x, y)` teilt standardmäßig durch $n - 1$ und liefert 1965, nicht 1637,5. Für die Konvention des Kapitels müssen Sie `np.cov(x, y, bias=True)` schreiben. Der Koeffizient $r$ hängt dagegen nicht von der Konvention ab, weil derselbe Faktor oben und unten auftritt (Lektion 1.5).

**c)** Es gibt keinen Widerspruch. Erstens misst $r$, wie eng sich die Punkte um *die* Gerade scharen, die am besten zu ihnen passt (die Kapitel 3 Ihnen zu berechnen beibringen wird), nicht um unsere nach Augenmaß gezeichnete Gerade. Zweitens ist Nähe relativ: Die Preise streuen mit einer Standardabweichung von etwa 72 k€, und Abweichungen von 5 bis 11 k€ im Betrag sind klein gegenüber dieser Streuung. Ein $r$ nahe 1 garantiert also keine kleinen Fehler in k€. Um die Größe der Fehler zu kennen, muss man sie berechnen, wie in Aufgabe 1.6.

*Erwartete Elemente:* „$r$ bezieht sich auf die beste Gerade“; „die Abweichungen sind klein im Verhältnis zur Streuung der Preise“; die Unterscheidung zwischen Ausrichtung und Größe der Fehler in k€.

In [ ]:
print('np.cov standardmäßig (Division durch n − 1):', np.cov(x, y)[0, 1])
print('np.cov mit bias=True (Division durch n)     :', np.cov(x, y, bias=True)[0, 1])

## Aufgabe 1.5 — Vorhersagen, dann prüfen

*Daten D1.1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1), Median und Interquartilsabstand (Lektion 1.3).* Eine sechste T2-Wohnung wird im Norden für 240 k€ verkauft.

**a)** Erwartete Vorhersage: Der Mittelwert steigt deutlich, weil er die 240 k€ auf alle Verkäufe verteilt. Der Median bewegt sich kaum, weil sich der neue Verkauf nur „hinter der Mitte“ einreiht. Die Standardabweichung steigt stark, weil eine quadrierte Abweichung von etwa 50 k€ die Summe dominiert. Der Interquartilsabstand ändert sich wenig, weil er nur von den mittleren Werten abhängt.

In [ ]:
nord_plus = nord + [240]
print('Nord:', sorted(nord))
print('Nord mit dem Verkauf zu 240 k€:', sorted(nord_plus))

**b)** Vervollständigen Sie die Funktion `resume`. Sie soll die vier Kennzahlen zurückgeben, mit der Standardabweichung durch $n$ geteilt und den Quartilen nach der Regel des Buchs berechnet (`quartiles_livre`, weiter oben bereitgestellt). Die Schlüssel des Wörterbuchs bleiben französisch: `moyenne` = Mittelwert, `médiane` = Median, `écart-type` = Standardabweichung, `écart interquartile` = Interquartilsabstand.

In [ ]:
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': np.mean(serie),
        'médiane': np.median(serie),
        'écart-type': np.std(serie, ddof=0),   # Division durch n
        'écart interquartile': q3 - q1,
    }

In [ ]:
tableau = pd.DataFrame({'Nord (5 Verk.)': resume(nord), 'Nord + 240 k€ (6 Verk.)': resume(nord_plus),
                        'Süd (5 Verk.)': resume(sud)})
if tableau.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k}: {'richtig' if ok else 'bitte prüfen'}")
tableau.round(2)

In [ ]:
# Die beiden Reihen nach dem Vorbild der Abbildung „Mittelwert ± Standardabweichung“ aus Lektion 1.1
if tableau.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion resume, um das Diagramm anzuzeigen.')
else:
    points_par_rangee({'Nord': nord, 'Nord + 240': nord_plus},
                      'Verkaufspreis einer T2-Wohnung (k€)',
                      bandes={'Nord': (np.mean(nord), np.std(nord, ddof=0)),
                              'Nord + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

### Lösung zu Aufgabe 1.5

**b) Auf Papier.** Mit dem Verkauf zu 240 k€ steigt die Summe von 900 auf 1140 k€ für 6 Verkäufe: Mittelwert **190 k€** (statt 180). Sortiert lauten die sechs Werte 174, 177, 181, 183, 185, 240. $n$ ist gerade, also beträgt der Median (181 + 183)/2 = **182 k€** (statt 181). Die Abweichungen von 190 sind −16, −13, −9, −7, −5, +50. Ihre Quadrate betragen 256, 169, 81, 49, 25 und 2500, mit einer Summe von 3080. Die Varianz beträgt 3080/6 ≈ 513,3 k€² und die Standardabweichung **≈ 22,66 k€** (statt 4). Quartile (Gleichung 1.5): vorher Ränge ⌈5/4⌉ = 2 und ⌈15/4⌉ = 4, also Q1 = 177 und Q3 = 183, daher ein Interquartilsabstand von **6 k€**. Nachher Ränge ⌈6/4⌉ = 2 und ⌈18/4⌉ = 5, also Q1 = 177 und Q3 = 185, daher **8 k€**.

**c)** Nein. Fünf von sechs Verkäufen liegen weiterhin höchstens 6 k€ von 180 k€ entfernt. Ein einziger außergewöhnlicher Verkauf genügt, um die Standardabweichung von 4 auf 22,66 k€ zu heben, über die 20 k€ des Südens, weil seine Abweichung von 50 k€ mit 2500 in eine Summe von 3080 eingeht. Die robusten Maße erzählen etwas anderes: Der Interquartilsabstand im Norden steigt von 6 auf 8 k€ und bleibt weit unter dem des Südens (30 k€). Für eine gewöhnliche T2-Wohnung im Norden bleiben die Preise sehr gut vorhersehbar. Die Standardabweichung zeigt vor allem einen Ausreißer-Verkauf an, den man gesondert betrachten muss.

*Erwartete Elemente:* die vier Werte vorher/nachher; der Gegensatz zwischen empfindlichen (Mittelwert, Standardabweichung) und robusten Kennzahlen (Median, Interquartilsabstand); eine differenzierte Schlussfolgerung.

## Aufgabe 1.6 — Herausforderung: zwei Geraden, derselbe mittlere Fehler

*Daten D0.1. Begriffe: vorzeichenbehaftete Abweichung und Betrag (Einleitung, Lektion 1.1), Standardabweichung (Lektion 1.1).* Nach Augenmaß gezeichnete Gerade: vorhergesagter Preis = 3 × Fläche + 30. Faule Gerade: vorhergesagter Preis = 222 k€ für alle Wohnungen.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = prix - (3 * surface + 30)
ecarts_paresseuse = prix - 222

In [ ]:
# Die drei Kennzahlen aus Frage b) (Standardabweichung: Division durch n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    indicateurs = pd.DataFrame({
        nom: {'Mittel vorzeichenbeh. Abweichungen': e.mean(),
              'Mittel der Abweichungsbeträge': np.abs(e).mean(),
              'Std.-abw. vorzeichenbeh. Abweich.': e.std(ddof=0)}
        for nom, e in [('Gerade nach Augenmaß', ecarts_oeil), ('faule Gerade', ecarts_paresseuse)]
    })
    print('Abweichungen, Gerade nach Augenmaß:', ecarts_oeil)
    print('Abweichungen, faule Gerade        :', ecarts_paresseuse)
    print('Prüfung:', 'richtig' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'bitte prüfen')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="Gerade nach Augenmaß")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='faule Gerade')
    ax.set_xlabel('Wohnfläche (m²)')
    ax.set_ylabel('Abweichung beobachtet − vorhergesagt (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

### Lösung zu Aufgabe 1.6

**a)** Abweichungen der faulen Geraden (beobachtet − 222): −104, −50, −37, +23, +57, +113 k€. Ihre Summe beträgt 2 und ihr Mittelwert ≈ **0,33 k€**. Das ist genau der Mittelwert der vorzeichenbehafteten Abweichungen der nach Augenmaß gezeichneten Geraden (−8, +7, −7, +5, −6, +11: Summe 2, Mittelwert ≈ 0,33 k€).

**b)**

| | Gerade nach Augenmaß | Faule Gerade |
|:--|--:|--:|
| Mittelwert der vorzeichenbehafteten Abweichungen | 0,33 k€ | 0,33 k€ |
| Mittelwert der Abweichungsbeträge | 44/6 ≈ 7,33 k€ | 384/6 = 64 k€ |
| Standardabweichung der vorzeichenbehafteten Abweichungen (Division durch $n$) | ≈ 7,56 k€ | ≈ 72,15 k€ |

Detail für die Gerade nach Augenmaß: Der Mittelwert beträgt 1/3. Die Abweichungen von diesem Mittelwert haben Quadrate mit der Summe 344 − 6 × (1/3)² ≈ 343,33. Die Varianz beträgt 343,33/6 ≈ 57,22 und die Standardabweichung ≈ 7,56 k€.

**c)** Die nach Augenmaß gezeichnete Gerade ist viel besser: Ihre Fehler betragen im Betrag etwa 7 k€, gegenüber 64 k€ bei der faulen Geraden. Der Mittelwert der Abweichungsbeträge und die Standardabweichung der vorzeichenbehafteten Abweichungen unterscheiden sie. Der Mittelwert der vorzeichenbehafteten Abweichungen schafft das nicht: Keine der beiden Geraden neigt zu einer Seite, und die Vorzeichen heben sich in beiden Fällen auf.

**d)** Die Abweichungen der faulen Geraden betragen $y_i - 222$. Das Abziehen derselben Konstanten verschiebt die ganze Reihe, ohne den Abstand jedes Werts zum Mittelwert zu ändern. Auch der Mittelwert sinkt um 222, also bleibt jede Abweichung vom Mittelwert $(y_i - 222) - (\bar{y} - 222) = y_i - \bar{y}$ gleich. Die Standardabweichung der Abweichungen ist daher die der Preise, ≈ 72,15 k€. Eine Gerade, die die Fläche ignoriert, irrt sich im Mittel um die Streuung der Preise selbst. Kapitel 3 wird diesen Vergleich nutzen.

*Erwartete Elemente:* die beiden gleichen Mittelwerte; zwei deutlich verschiedene Kennzahlen für die Fehlergröße; die Erklärung durch das Aufheben der Vorzeichen; das Verschiebungsargument für d).

In [ ]:
# Kontrolle zu d): Die Standardabweichung der Abweichungen der faulen Geraden ist gleich der der Preise
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))